# Generación de OBD-II sintético

**Fase:** E2

## Objetivo

Producir señales OBD-II plausibles y reproducibles, con relaciones físicas aproximadas entre variables y escenarios de fallo controlados.

## Relación con la pregunta de investigación

> ¿Puede un agente con RAG explicar el diagnóstico de un vehículo a partir de datos OBD-II reales, rastreando cada explicación hasta su evidencia de origen, y reconocer cuándo la evidencia disponible no es suficiente para sostener una explicación en lugar de generar una hipótesis sin fundamento?

Aporta el control experimental: permite conocer la verdad de referencia y manipular la evidencia disponible sin ambigüedad.

## Hipótesis

_Por declarar antes de ejecutar la fase._

## Datos

_Dataset, versión y partición utilizados._

## Método

_Procedimiento, parámetros y semillas._

## Métricas

_Qué se mide y con qué instrumento._

## Criterio de éxito o refutación

_Fijado antes de mirar los resultados._


In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

from mecanico_ia_exp.config import paths


## Las constantes del generador salen del manual

Cada valor lleva su cita. Un generador con constantes inventadas es coherente consigo mismo y no corresponde a ningún vehículo.

In [2]:
import yaml

calib = yaml.safe_load((paths.CONFIGS / 'calibracion_mazda3_bk.yaml').read_text())
for nombre, p in calib['parametros'].items():
    fuente = p.get('fuente')
    print(f"{nombre:<28} {str(fuente)[:44]}")

ralenti_rpm                  srvc/B3E011002000W01.pdf
termostato_apertura_c        srvc/B3E010002000T03.pdf
ect_sensor_tension           srvc/B3E014018880W08.pdf
ventilador_electrico_c       srvc/B3E014000140T45.pdf
maf_tension_v                ['srvc/B3E010200100W01.pdf', 'srvc/B3E010200
maf_caudal_max_g_s           srvc/B3E010200100W08.pdf
avance_encendido_btdc        srvc/B3E011002000W01.pdf


## Comprobación: ¿el generador respeta la especificación?

In [3]:
from mecanico_ia_exp.obd.synthetic import SessionSpec, generate_session

d = generate_session(SessionSpec(session_id='demo', regime='urban',
                                 duration_s=900.0, seed=5))
ral = d[d['vehicle_speed'] < 2.0]
print(f"ralentí        {ral['rpm'].min():.0f}-{ral['rpm'].max():.0f} rpm      "
      f"(manual: 640-740)")
print(f"caudal ralentí {ral['maf'].mean():.2f} g/s            (manual: 2,5)")
print(f"refrigerante   {d['coolant_temperature'].max():.1f} °C           "
      f"(manual: apertura 80-84)")
print(f"avance ralentí {ral['ignition_timing'].mean():.1f} BTDC          "
      f"(manual: 9-11)")

ralentí        646-721 rpm      (manual: 640-740)
caudal ralentí 2.56 g/s            (manual: 2,5)
refrigerante   82.5 °C           (manual: apertura 80-84)
avance ralentí 10.0 BTDC          (manual: 9-11)


## Las cinco condiciones y su respaldo documental

In [4]:
from mecanico_ia_exp.obd.faults import FAULTS

for fid, f in FAULTS.items():
    print(f"{fid}")
    print(f"    señales : {', '.join(f.affected_signals)}")
    print(f"    patrón  : {f.temporal_pattern[:70]}")

F01_vacuum_leak
    señales : stft_bank1, ltft_bank1, map, rpm
    patrón  : persistente, acentuado en ralentí; el ajuste a largo plazo sube en ram
F02_maf_drift
    señales : maf, stft_bank1, ltft_bank1
    patrón  : deriva progresiva con rampa de instalación de ~5 min
F03_thermostat_stuck
    señales : coolant_temperature
    patrón  : techo constante de temperatura; la firma es la ausencia de calentamien
F04_o2_lag
    señales : oxygen_sensor, stft_bank1
    patrón  : persistente; amortiguación de la oscilación característica
F05_misfire
    señales : rpm, engine_load, stft_bank1, oxygen_sensor
    patrón  : episódico: eventos de 1 a 3 s con tasa de aparición baja


## Resultados

_Lo observado, sin interpretación._

## Interpretación

_Qué significa, y qué no significa._

## Limitaciones

_Qué queda fuera del alcance de lo mostrado._

## Conclusión experimental

_Respuesta al criterio fijado en la cabecera._

## Artefactos generados

_Ficheros producidos, con su ruta._
